# Agents Lab: Ingest Test Cases (Quality Center & Excel) + Generate Selenium Scripts with Repo Context

### Goal: Fetch/normalize test cases, retrieve similar existing scripts from a git repo, and generate new Selenium tests with context (open‑source LLM).


This lab contains two cooperating agents:

1. **Ingestion Agent** — fetches test cases from **HP ALM / Quality Center** (REST scaffold + mock) or **Excel/CSV**, and normalizes into a canonical schema.
2. **Authoring Agent** — given a *new* test case, retrieves similar **existing Selenium scripts** from a repo and **generates a new Selenium test** (Python + pytest by default) via a small open-source LLM.

Now includes a **Public Data Fast Start** so the lab is fully runnable without internal systems.


## 0) Public Data Fast Start (no internal systems needed)


Use **public repos** as context and (optionally) pull **public test-case spreadsheets**.

- **Context Repos (examples):** Any public Selenium test repo (Python/Java). Paste URLs below.
- **Demo Apps:** `https://www.saucedemo.com` and `https://the-internet.herokuapp.com/` are commonly used practice apps.
- **Public Test Cases:** You can paste a direct CSV/XLSX URL to download and map.

> If internet is restricted, skip this and use the built-in sample Excel + sample repo created later.


In [1]:

# Choose public repos & optional public Excel URL.
# Paste one URL per line when prompted. Leave blank to skip.
# Examples (paste your own trusted links that contain Selenium tests):
#   https://github.com/venkywarriors/selenium_with_python.git
#
# Optional: paste a direct CSV/XLSX URL for public test cases.
#   e.g., https://raw.githubusercontent.com/<org>/<repo>/main/testcases.csv

import os, subprocess, sys, zipfile, urllib.request, pandas as pd
from pathlib import Path

os.makedirs("public_fast_start", exist_ok=True)
os.makedirs("repo", exist_ok=True)
os.makedirs("uploads", exist_ok=True)

print("Enter Git repo URLs (blank line to finish):")
urls = []
while True:
    u = input().strip()
    if not u: break
    urls.append(u)

for i, u in enumerate(urls):
    try:
        print(f"Cloning {u} ...")
        subprocess.run(["git","clone","--depth","1",u,f"repo/public_{i}"], check=True)
    except Exception as e:
        print("Clone failed:", e)

# Optional public test-case sheet
print("Optional: paste a direct CSV/XLSX URL for test cases (press Enter to skip):")
sheet_url = input().strip()
if sheet_url:
    dest = Path("public_fast_start/public_testcases")
    dest.parent.mkdir(parents=True, exist_ok=True)
    # naive extension detection
    ext = ".csv" if sheet_url.lower().endswith(".csv") else ".xlsx"
    outp = str(dest) + ext
    try:
        print("Downloading public test cases ...")
        urllib.request.urlretrieve(sheet_url, outp)
        print("Saved to:", outp)
    except Exception as e:
        print("Download failed:", e)
else:
    print("No public sheet provided; you can upload Excel/CSV in Section 2.")

# Paste one URL per line when prompted. Leave blank to skip.
# Examples (paste your own trusted links that contain Selenium tests):
#   https://github.com/venkywarriors/selenium_with_python.git

Enter Git repo URLs (blank line to finish):
https://github.com/venkywarriors/selenium_with_python.git

Cloning https://github.com/venkywarriors/selenium_with_python.git ...
Optional: paste a direct CSV/XLSX URL for test cases (press Enter to skip):

No public sheet provided; you can upload Excel/CSV in Section 2.


## Theory: Why two agents & canonical schema


**Two-agent design** gives deterministic ingestion + generative authoring grounded on repo code.

**Canonical fields** we use across sources:
```
[id, title, steps, expected, component, priority, tags]
```


## 1) Setup

In [2]:

# !pip -q install --upgrade pip
!pip -q install pandas openpyxl numpy requests python-dotenv tqdm scikit-learn
!pip -q install transformers accelerate bitsandbytes

import os, sys, json, re, glob, io, zipfile, shutil, textwrap, subprocess
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm

# LLM imports
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 8.7 MB/s eta 0:00:00
Device: cuda


## 2) Ingestion Agent: Quality Center (ALM) + Excel/CSV


- **Excel/CSV upload** — map flexible headers to canonical fields.
- **Quality Center / ALM (REST scaffold)** — set env vars; toggle `MOCK_ALM=True` for public runs.
- **Public sheet (if downloaded in 0)** — we will auto-include it if present.


In [3]:

import os, json, requests
from typing import List
from google.colab import files

CANON_COLS = ["id","title","steps","expected","component","priority","tags"]

def from_excel_or_csv(paths: List[str]) -> pd.DataFrame:
    frames = []
    for p in paths:
        p = p.strip()
        if not os.path.isfile(p):
            continue
        if p.lower().endswith(".xlsx"):
            df = pd.read_excel(p)
        else:
            df = pd.read_csv(p)
        cols = {c.lower(): c for c in df.columns}
        def pick(*names):
            for n in names:
                if n.lower() in cols: return cols[n.lower()]
            return None
        idc = pick("id","test id","tc_id")
        titlec = pick("title","name","test name")
        stepsc = pick("steps","step","test steps","procedure")
        expectc = pick("expected","expected result","expected_results")
        compc = pick("component","module","area")
        prio = pick("priority","prio","p")
        tagc = pick("tags","label","labels","category")

        out = pd.DataFrame()
        out["id"]        = df[idc] if idc else pd.Series([None]*len(df))
        out["title"]     = df[titlec] if titlec else pd.Series(["Untitled"]*len(df))
        out["steps"]     = df[stepsc] if stepsc else ""
        out["expected"]  = df[expectc] if expectc else ""
        out["component"] = df[compc] if compc else ""
        out["priority"]  = df[prio] if prio else ""
        out["tags"]      = df[tagc] if tagc else ""
        frames.append(out[CANON_COLS])
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=CANON_COLS)

def from_quality_center(mock: bool = False) -> pd.DataFrame:
    if mock:
        with open("sample_data/alm_sample_tests.json","r") as f:
            data = json.load(f)
        return pd.DataFrame(data)[CANON_COLS]

    base = os.getenv("ALM_BASE_URL")
    domain = os.getenv("ALM_DOMAIN")
    project = os.getenv("ALM_PROJECT")
    user = os.getenv("ALM_USER")
    pwd = os.getenv("ALM_PASS")
    if not all([base, domain, project, user, pwd]):
        raise RuntimeError("Set ALM_* env vars or enable mock=True.")

    sess = requests.Session()
    auth_url = f"{base}/authentication-point/authenticate"
    r = sess.post(auth_url, auth=(user, pwd), verify=True)
    if r.status_code not in [200, 201, 204]:
        raise RuntimeError(f"ALM auth failed: {r.status_code} {r.text[:200]}")

    tests_url = f"{base}/rest/domains/{domain}/projects/{project}/tests"
    r = sess.get(tests_url)
    if r.status_code != 200:
        raise RuntimeError(f"ALM tests fetch failed: {r.status_code} {r.text[:200]}")

    items = []
    payload = r.json() if r.headers.get("Content-Type","").startswith("application/json") else {}
    for t in payload.get("entities", []):
        fields = {f["Name"]: f["values"][0]["value"] for f in t.get("Fields",[]) if f.get("values")}
        items.append({
            "id": fields.get("id") or fields.get("test-id"),
            "title": fields.get("name") or fields.get("test-name"),
            "steps": fields.get("steps",""),
            "expected": fields.get("expected",""),
            "component": fields.get("component",""),
            "priority": fields.get("priority",""),
            "tags": fields.get("tags",""),
        })
    if not items:
        items = [{"id": None,"title":"(ALM item)","steps":"","expected":"","component":"","priority":"","tags":""}]
    return pd.DataFrame(items)[CANON_COLS]

# Upload local Excel/CSV (optional)
print("Upload Excel/CSV (optional)")
uploaded = files.upload()
uploaded_paths = []
for fname in uploaded.keys():
    p = f"uploads/{fname}"
    os.makedirs("uploads", exist_ok=True)
    with open(p,"wb") as f:
        f.write(uploaded[fname])
    uploaded_paths.append(p)
print("Uploaded:", uploaded_paths)

# Include public sheet if fetched in step 0
public_sheet = None
for ext in (".csv",".xlsx"):
    candidate = f"public_fast_start/public_testcases{ext}"
    if os.path.isfile(candidate):
        public_sheet = candidate
        break

paths = uploaded_paths + ([public_sheet] if public_sheet else [])
df_upload = from_excel_or_csv(paths) if paths else pd.DataFrame(columns=CANON_COLS)

# Mock ALM sample for public runs
MOCK_ALM = True
os.makedirs("sample_data", exist_ok=True)
if not os.path.isfile("sample_data/alm_sample_tests.json"):
    sample = [
        {"id":"ALM-101","title":"Login with valid user","steps":"1. Open login\\n2. Enter valid creds\\n3. Click Sign in","expected":"Home loads","component":"Auth","priority":"P1","tags":"smoke,login"},
        {"id":"ALM-102","title":"Login invalid password","steps":"1. Open login\\n2. Enter wrong pwd\\n3. Click Sign in","expected":"Error toast","component":"Auth","priority":"P1","tags":"negative,login"},
    ]
    with open("sample_data/alm_sample_tests.json","w") as f:
        json.dump(sample, f, indent=2)

df_alm = from_quality_center(mock=MOCK_ALM)

df_cases = pd.concat([df_upload, df_alm], ignore_index=True).dropna(how="all")
print("Ingested rows:", len(df_cases))
df_cases.head(10)
# upload the sample test cases excel file here

Upload Excel/CSV (optional)


Saving sample_testcases.xlsx to sample_testcases.xlsx
Uploaded: ['uploads/sample_testcases.xlsx']
Ingested rows: 5


,id,title,steps,expected,component,priority,tags
0,TC-001,User can reset password,1. Go to /forgot\n2. Enter email\n3. Submit,Reset email sent,Auth,P1,"smoke,regression"
1,TC-002,Add to cart updates subtotal,1. Open product\n2. Click Add to Cart\n3. View...,Subtotal increases by price,Cart,P1,"cart,ui"
2,TC-003,Empty cart shows CTA,1. Open cart\n2. Remove all items,CTA to continue shopping visible,Cart,P2,"cart,ui,edge"
3,ALM-101,Login with valid user,1. Open login\n2. Enter valid creds\n3. Click ...,Home loads,Auth,P1,"smoke,login"
4,ALM-102,Login invalid password,1. Open login\n2. Enter wrong pwd\n3. Click Si...,Error toast,Auth,P1,"negative,login"


## 3) Build Repo Context Index (existing Selenium tests)

Clone public repos you chose in step 0 (already cloned to `repo/public_*`) or upload ZIP here. We’ll index `.py/.java`.

In [4]:

from google.colab import files
import zipfile, os, subprocess
from pathlib import Path
import pandas as pd

# Upload an additional ZIP (optional)
print("Upload a ZIP with tests (optional):")
uploaded = files.upload()
for fname in uploaded.keys():
    p = f"repo/{fname}"
    with open(p,"wb") as f:
        f.write(uploaded[fname])
    if p.lower().endswith(".zip"):
        with zipfile.ZipFile(p,"r") as z:
            z.extractall("repo")
        os.remove(p)

# Fallback sample repo content if empty
def ensure_sample_repo():
    py_tests_dir = Path("repo/sample_pytests")
    if not py_tests_dir.exists():
        py_tests_dir.mkdir(parents=True, exist_ok=True)
        py_tests_dir.joinpath("test_login.py").write_text('''
import pytest
from selenium import webdriver
from selenium.webdriver.common.by import By

def test_login_valid_user():
    driver = webdriver.Chrome()
    driver.get("https://example.com/login")
    driver.find_element(By.ID,"username").send_keys("demo")
    driver.find_element(By.ID,"password").send_keys("demo123")
    driver.find_element(By.ID,"signin").click()
    assert "Dashboard" in driver.title
    driver.quit()
''')
        py_tests_dir.joinpath("test_login_negative.py").write_text('''
from selenium import webdriver
from selenium.webdriver.common.by import By

def test_login_invalid_password():
    driver = webdriver.Chrome()
    driver.get("https://example.com/login")
    driver.find_element(By.ID,"username").send_keys("demo")
    driver.find_element(By.ID,"password").send_keys("wrong")
    driver.find_element(By.ID,"signin").click()
    error = driver.find_element(By.CSS_SELECTOR,".toast-error").text
    assert "Invalid" in error
    driver.quit()
''')

# If no files found in repo/, drop in the sample
paths = []
for ext in ("*.py","*.java"):
    paths += [str(p) for p in Path("repo").rglob(ext)]
if not paths:
    ensure_sample_repo()
    paths = []
    for ext in ("*.py","*.java"):
        paths += [str(p) for p in Path("repo").rglob(ext)]
print("Files found:", len(paths))

def read_text_safe(p):
    try:
        return Path(p).read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""

docs = []
for p in paths:
    txt = read_text_safe(p)
    if not txt.strip():
        continue
    header = "\\n".join(txt.splitlines()[:20])
    name = Path(p).name
    docs.append({"path": p, "name": name, "text": txt, "header": header})

df_repo = pd.DataFrame(docs)
print("Indexed repo files:", len(df_repo))
df_repo.head(5)
# press click cancel upload

Upload a ZIP with tests (optional):


Files found: 185
Indexed repo files: 157


,path,name,text,header
0,repo/public_0/Python_basics/ObjectOrientedProg...,043 classdemo1.py,"""""""\nObject Oriented Programming\n""""""\n\ns = ""...","""""""\nObject Oriented Programming\n""""""\n\ns = ""..."
1,repo/public_0/Python_basics/ObjectOrientedProg...,053 modules-car.py,"""""""\nThis is our own module which does not exi...","""""""\nThis is our own module which does not exi..."
2,repo/public_0/Python_basics/ObjectOrientedProg...,047 classdemo-inheritance2.py,"'''\nCreated on May 30, 2018\n@author: venkate...","'''\nCreated on May 30, 2018\n@author: venkate..."
3,repo/public_0/Python_basics/ObjectOrientedProg...,046 classdemo-inheritance1.py,"'''\nCreated on May 30, 2018\n@author: venkate...","'''\nCreated on May 30, 2018\n@author: venkate..."
4,repo/public_0/Python_basics/ObjectOrientedProg...,044 classdemo2.py,"""""""\nObject Oriented Programming\n""""""\n\nclass...","""""""\nObject Oriented Programming\n""""""\n\nclass..."


## 4) Retrieval: find similar tests for a new test case

In [5]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def build_index(df_repo: pd.DataFrame):
    corpus = df_repo["text"].tolist()
    vec = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
    mat = vec.fit_transform(corpus)
    return vec, mat

def retrieve_similar(vec, mat, df_repo, query: str, k=5):
    qv = vec.transform([query])
    sims = cosine_similarity(qv, mat).ravel()
    idx = np.argsort(-sims)[:k]
    results = df_repo.iloc[idx][["path","name","text"]].copy()
    results["score"] = sims[idx]
    return results

vectorizer, matrix = build_index(df_repo)

# New test case (edit freely)
print("Enter a new test case title (or press Enter to use a sample):")
user_title = input().strip() or "Checkout applies discount code at payment"
user_steps = "1. Add item to cart\\n2. Go to checkout\\n3. Apply code SAVE10\\n4. Complete payment"
user_expected = "Total reflects discount; confirmation page shows coupon applied"

query = f"{user_title}\\n{user_steps}\\nExpected: {user_expected}"
topk = retrieve_similar(vectorizer, matrix, df_repo, query, k=5)
topk
# press enter without typing anything, it will take a sample title

Enter a new test case title (or press Enter to use a sample):
Demo test case


,path,name,text,score
151,repo/public_0/Python_basics/MethodsWorkingWith...,039 methodsdemo3.py,"""""""\nPositional Parameters\nThey are like opti...",0.179530
148,repo/public_0/Python_basics/MethodsWorkingWith...,037 methodsdemo1.py,"""""""\nA group of code statements which can perf...",0.165494
153,repo/public_0/Python_basics/MethodsWorkingWith...,038 methodsdemo2.py,"""""""\nA group of code statements which can perf...",0.152826
32,repo/public_0/Python_basics/utilities/checkpoi...,checkpoint.py,"""""""\n@package utilities\n\nCheckPoint class im...",0.083806
144,repo/public_0/Python_basics/AutomationFramewor...,175 2-teststatus.py,"""""""\n@package utilities\n\nCheckPoint class im...",0.079538


## 5) Authoring Agent: generate Selenium (pytest, open-source LLM)

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

def load_llm(model_id=MODEL_ID):
    print(f"Loading model: {model_id}")
    kwargs = {}
    if device == "cuda":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        kwargs = dict(
            device_map="auto",
            quantization_config=bnb_config,
        )
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, **kwargs)
    if device == "cpu":
        mdl = mdl.to("cpu")
    return tok, mdl

tokenizer, model = load_llm()
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

def load_llm(model_id=MODEL_ID):
    print(f"Loading model: {model_id}")
    kwargs = {}
    if device == "cuda":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        kwargs = dict(
            device_map="auto",
            quantization_config=bnb_config,
        )
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, **kwargs)
    if device == "cpu":
        mdl = mdl.to("cpu")
    return tok, mdl

tokenizer, model = load_llm()

def gen(text: str, max_new_tokens=700, temperature=0.2):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=temperature, do_sample=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

def build_prompt(new_title, new_steps, new_expected, similar_df: pd.DataFrame, language="python", target_app="https://www.saucedemo.com"):
    snippets = []
    for _, r in similar_df.iterrows():
        body = r["text"]
        if len(body) > 2000:
            body = body[:2000]
        snippets.append(f"### FILE: {r['path']}\\n{body}")
    context_block = "\\n\\n".join(snippets)

    return f'''
You are a senior QA automation engineer. Generate a runnable Selenium test in {language} (pytest style).
Rules:
- Target app (hint): {target_app}
- Use Selenium WebDriver best practices and explicit waits where suitable.
- Keep locators reasonable; avoid brittle absolute XPaths.
- Include setup/teardown to close the driver.
- Add clear assertions for the expected outcome.
- Reuse style & naming patterns seen in the context.
- Return only code, no explanation.

NEW TEST CASE
Title: {new_title}
Steps:
{new_steps}
Expected:
{new_expected}

CONTEXT (existing repo snippets):
{context_block}
'''

print("Choose target app (press Enter for SauceDemo):")
target_app = input().strip() or "https://www.saucedemo.com"
prompt = build_prompt(user_title, user_steps, user_expected, topk, language="python", target_app=target_app)
selenium_code = gen(prompt, max_new_tokens=700, temperature=0.15)
print(selenium_code)


# press enter when a text box opens with "Choose target app (press Enter for SauceDemo):"

Loading model: Qwen/Qwen2.5-1.5B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading model: Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Choose target app (press Enter for SauceDemo):


You are a senior QA automation engineer. Generate a runnable Selenium test in python (pytest style).
Rules:
- Target app (hint): https://www.saucedemo.com
- Use Selenium WebDriver best practices and explicit waits where suitable.
- Keep locators reasonable; avoid brittle absolute XPaths.
- Include setup/teardown to close the driver.
- Add clear assertions for the expected outcome.
- Reuse style & naming patterns seen in the context.
- Return only code, no explanation.

NEW TEST CASE
Title: Demo test case
Steps:
1. Add item to cart\n2. Go to checkout\n3. Apply code SAVE10\n4. Complete payment
Expected:
Total reflects discount; confirmation page shows coupon applied

CONTEXT (existing repo snippets):
### FILE: repo/public_0/Python_basics/MethodsWorkingWithReusableCode/039 methodsdemo3.py\n"""
Positional Parameters
They are like optional parameters
And can be assigned a default value, if no value is provided from outside
"""

def sum_nums(n

## 6) Save output & export summary

In [7]:

from pathlib import Path
from datetime import datetime

out_dir = Path("generated_tests")
out_dir.mkdir(exist_ok=True, parents=True)

ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
out_path = out_dir / f"test_generated_{ts}.py"

with open(out_path, "w", encoding="utf-8") as f:
    f.write(selenium_code if selenium_code.strip() else "# (empty)")

print("Saved:", out_path)

# Summary artifact
md_lines = []
md_lines.append(f"# Test Authoring Summary")
md_lines.append(f"_Generated: {datetime.utcnow().isoformat()}Z_")
md_lines.append("")
md_lines.append("## New Test Case")
md_lines.append(f"**Title:** {user_title}")
md_lines.append("**Steps:**")
for line in user_steps.split("\\n"):
    md_lines.append(f"- {line}")
md_lines.append(f"**Expected:** {user_expected}")
md_lines.append("")
md_lines.append("## Retrieved Context (Top-K)")
for _, r in topk.iterrows():
    md_lines.append(f"- `{r['path']}` (score={r['score']:.3f})")
md_lines.append("")
md_lines.append("## Generated Selenium (preview)")
md_lines.append("```python")
preview = (selenium_code or "").strip()
if len(preview) > 900:
    preview = preview[:900] + "\\n# ...truncated..."
md_lines.append(preview if preview else "# (empty)")
md_lines.append("```")

art_dir = Path("artifacts")
art_dir.mkdir(exist_ok=True, parents=True)
md_path = art_dir / "authoring_summary.md"
md_path.write_text("\\n".join(md_lines), encoding="utf-8")
print("Summary:", md_path)


Saved: generated_tests/test_generated_20260506_135159.py
Summary: artifacts/authoring_summary.md


/tmp/ipykernel_1098/1169681156.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
/tmp/ipykernel_1098/1169681156.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  md_lines.append(f"_Generated: {datetime.utcnow().isoformat()}Z_")


## 7) Next steps


- Map ALM fields to your instance (endpoints vary).
- Add Page Objects and prompt the model to **extend POMs** instead of raw locators.
- Switch TF‑IDF to embeddings for better semantic retrieval.
- Add linters/formatters & CI to auto-propose tests on new stories.
